# Model Evaluation for ECoG Classification

Comparing different approaches for auditory stimulus classification:

**From-scratch models:**
1. Simple MLP (baseline)
2. Small 1D CNN
3. Tiny Transformer

**Established architectures:**
4. EEGNet (Lawhern et al. 2018) - designed for BCI
5. PatchTST-style model - time series transformer

**Constraints:**
- Model size: <25MB (ideally <5MB for best score)
- Latency: <500ms (ideally <100ms)
- Must support streaming inference

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy import signal
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import time

import sys
sys.path.insert(0, str(Path.cwd().parent))

from brainstorm.download import download_train_validation_data
from brainstorm.loading import load_raw_data, load_channel_coordinates
from brainstorm.constants import N_CHANNELS, SAMPLING_RATE

plt.style.use('fivethirtyeight')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load data
DATA_PATH = Path("./data")

if not DATA_PATH.exists() or not any(DATA_PATH.glob("*.parquet")):
    print("Downloading data...")
    download_train_validation_data()

train_features, train_labels = load_raw_data(DATA_PATH, step="train")
val_features, val_labels = load_raw_data(DATA_PATH, step="validation")

print(f"Train: {train_features.shape}, Val: {val_features.shape}")
print(f"Unique labels: {sorted(train_labels['label'].unique())}")

## Preprocessing Approaches

### Approach 1: Raw + PCA (simple baseline)

In [ ]:
from sklearn.decomposition import PCA

# PCA for channel reduction
n_components = 64
pca = PCA(n_components=n_components)

X_train = train_features.values
X_val = val_features.values
y_train = train_labels['label'].values
y_val = val_labels['label'].values

# Fit PCA on training data
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

print(f"Explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print(f"Reduced shape: {X_train_pca.shape}")

### Approach 2: Spectrogram Features (Quilee's idea)

Convert time series to frequency representation using short-time FFT.

In [ ]:
def compute_spectrogram_features(data, fs=1000, nperseg=128, noverlap=64):
    """
    Compute spectrogram features for each channel.
    
    For streaming: we use a single window centered on current sample.
    Returns power in frequency bands relevant to neural signals.
    """
    n_samples, n_channels = data.shape
    
    # Define frequency bands of interest
    bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 100),
        'high_gamma': (100, 200),
    }
    
    n_bands = len(bands)
    features = np.zeros((n_samples, n_channels * n_bands))
    
    # For each channel, compute band powers using welch on sliding windows
    for ch in range(min(n_channels, 100)):  # Limit for demo
        for i, (band_name, (low, high)) in enumerate(bands.items()):
            # Simple bandpass + envelope as fast approximation
            nyq = fs / 2
            b, a = signal.butter(4, [low/nyq, min(high/nyq, 0.99)], btype='band')
            filtered = signal.filtfilt(b, a, data[:, ch])
            # Envelope via Hilbert transform
            envelope = np.abs(signal.hilbert(filtered))
            features[:, ch * n_bands + i] = envelope
    
    return features

# Demo on subset
print("Computing spectrogram features (subset for demo)...")
subset_size = 5000
spec_features = compute_spectrogram_features(X_train[:subset_size, :100])
print(f"Spectrogram features shape: {spec_features.shape}")

### Approach 3: Windowed Input for Sequence Models

Create sliding windows for temporal context.

In [ ]:
def create_windows(X, y, window_size=128, stride=1):
    """
    Create sliding windows from time series data.
    
    Args:
        X: (n_samples, n_features)
        y: (n_samples,)
        window_size: number of timesteps per window
        stride: step between windows
    
    Returns:
        X_windows: (n_windows, window_size, n_features)
        y_windows: (n_windows,) - label at end of window
    """
    n_samples = X.shape[0]
    n_windows = (n_samples - window_size) // stride + 1
    
    X_windows = np.zeros((n_windows, window_size, X.shape[1]))
    y_windows = np.zeros(n_windows)
    
    for i in range(n_windows):
        start = i * stride
        end = start + window_size
        X_windows[i] = X[start:end]
        y_windows[i] = y[end - 1]  # Label at end of window (causal)
    
    return X_windows, y_windows

# Create windows from PCA-reduced data
window_size = 128  # 128ms at 1000Hz
X_train_win, y_train_win = create_windows(X_train_pca, y_train, window_size)
X_val_win, y_val_win = create_windows(X_val_pca, y_val, window_size)

print(f"Windowed train: {X_train_win.shape}")
print(f"Windowed val: {X_val_win.shape}")

## Model Architectures

### Model 1: Simple MLP (Baseline)

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size // 2, n_classes)
        )
    
    def forward(self, x):
        return self.net(x)

# Model size estimation
n_classes = len(np.unique(y_train))
mlp = SimpleMLP(n_components, 256, n_classes)
mlp_params = sum(p.numel() for p in mlp.parameters())
mlp_size_mb = mlp_params * 4 / (1024 * 1024)  # float32
print(f"MLP parameters: {mlp_params:,}")
print(f"MLP size: {mlp_size_mb:.3f} MB")

### Model 2: Small 1D CNN

In [ ]:
class SmallCNN(nn.Module):
    """Compact 1D CNN for windowed time series."""
    
    def __init__(self, n_channels, n_classes, window_size):
        super().__init__()
        
        self.conv1 = nn.Conv1d(n_channels, 32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(64)
        self.pool3 = nn.AdaptiveAvgPool1d(1)
        
        self.fc = nn.Linear(64, n_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # x: (batch, time, channels) -> (batch, channels, time)
        x = x.transpose(1, 2)
        
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        
        x = x.squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

cnn = SmallCNN(n_components, n_classes, window_size)
cnn_params = sum(p.numel() for p in cnn.parameters())
cnn_size_mb = cnn_params * 4 / (1024 * 1024)
print(f"CNN parameters: {cnn_params:,}")
print(f"CNN size: {cnn_size_mb:.3f} MB")

### Model 3: Tiny Transformer

In [ ]:
class TinyTransformer(nn.Module):
    """Minimal transformer for sequence classification."""
    
    def __init__(self, input_dim, n_classes, d_model=64, nhead=4, num_layers=2, max_len=256):
        super().__init__()
        
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # Learnable positional encoding
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Linear(d_model, n_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        seq_len = x.size(1)
        
        x = self.input_proj(x)
        x = x + self.pos_embedding[:, :seq_len, :]
        
        x = self.transformer(x)
        
        # Use last token for classification (causal)
        x = x[:, -1, :]
        return self.fc(x)

transformer = TinyTransformer(n_components, n_classes, d_model=64, nhead=4, num_layers=2)
trans_params = sum(p.numel() for p in transformer.parameters())
trans_size_mb = trans_params * 4 / (1024 * 1024)
print(f"Transformer parameters: {trans_params:,}")
print(f"Transformer size: {trans_size_mb:.3f} MB")

### Model 4: EEGNet (Established BCI Architecture)

EEGNet is a compact CNN designed specifically for EEG/neural signal classification.
Uses depthwise separable convolutions for efficiency.

Reference: Lawhern et al. (2018) "EEGNet: A Compact Convolutional Neural Network for EEG-based Brain-Computer Interfaces"

In [ ]:
class EEGNetModule(nn.Module):
    """
    EEGNet architecture adapted for our data.
    
    Original paper: Lawhern et al. (2018)
    Key features:
    - Temporal convolution to learn frequency filters
    - Depthwise convolution to learn spatial filters  
    - Separable convolution for feature combination
    - Very compact (~2-10K parameters typically)
    """
    
    def __init__(self, n_channels, n_classes, n_times, 
                 F1=8, D=2, F2=16, kernel_length=64, dropout=0.25):
        super().__init__()
        
        self.F1 = F1
        self.F2 = F2
        self.D = D
        
        # Block 1: Temporal convolution
        self.conv1 = nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)
        
        # Block 2: Depthwise convolution (spatial filter)
        self.depthwise = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        self.drop1 = nn.Dropout(dropout)
        
        # Block 3: Separable convolution
        self.separable1 = nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False)
        self.separable2 = nn.Conv2d(F1 * D, F2, (1, 1), bias=False)
        self.bn3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))
        self.drop2 = nn.Dropout(dropout)
        
        # ELU activation
        self.elu = nn.ELU()
        
        # Calculate output size
        with torch.no_grad():
            x = torch.zeros(1, 1, n_channels, n_times)
            x = self._forward_features(x)
            self.flat_size = x.numel()
        
        self.fc = nn.Linear(self.flat_size, n_classes)
    
    def _forward_features(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.bn1(x)
        
        # Block 2
        x = self.depthwise(x)
        x = self.bn2(x)
        x = self.elu(x)
        x = self.pool1(x)
        x = self.drop1(x)
        
        # Block 3
        x = self.separable1(x)
        x = self.separable2(x)
        x = self.bn3(x)
        x = self.elu(x)
        x = self.pool2(x)
        x = self.drop2(x)
        
        return x
    
    def forward(self, x):
        # Input: (batch, time, channels) -> (batch, 1, channels, time)
        x = x.transpose(1, 2).unsqueeze(1)
        x = self._forward_features(x)
        x = x.flatten(1)
        return self.fc(x)

# Create EEGNet
eegnet = EEGNetModule(n_components, n_classes, window_size, F1=8, D=2, F2=16, kernel_length=32)
eegnet_params = sum(p.numel() for p in eegnet.parameters())
eegnet_size_mb = eegnet_params * 4 / (1024 * 1024)
print(f"EEGNet parameters: {eegnet_params:,}")
print(f"EEGNet size: {eegnet_size_mb:.3f} MB")

### Model 5: PatchTST-style (Time Series Transformer)

PatchTST divides time series into patches and processes them with a transformer.
This approach has shown strong results on time series forecasting benchmarks.

Reference: Nie et al. (2023) "A Time Series is Worth 64 Words: Long-term Forecasting with Transformers"

In [ ]:
class PatchTST(nn.Module):
    """
    PatchTST-style architecture for time series classification.
    
    Key ideas from the paper:
    - Divide time series into patches (like ViT does for images)
    - Channel independence (process each channel separately, then combine)
    - Learnable positional embeddings
    
    Adapted for classification instead of forecasting.
    """
    
    def __init__(self, n_channels, n_classes, seq_len, 
                 patch_len=16, stride=8, d_model=64, nhead=4, 
                 num_layers=2, dropout=0.1):
        super().__init__()
        
        self.patch_len = patch_len
        self.stride = stride
        self.n_channels = n_channels
        
        # Number of patches
        self.n_patches = (seq_len - patch_len) // stride + 1
        
        # Patch embedding (linear projection of each patch)
        self.patch_embed = nn.Linear(patch_len, d_model)
        
        # Channel embedding (to distinguish channels)
        self.channel_embed = nn.Parameter(torch.randn(1, n_channels, 1, d_model) * 0.02)
        
        # Positional embedding for patches
        self.pos_embed = nn.Parameter(torch.randn(1, 1, self.n_patches, d_model) * 0.02)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model * n_channels, n_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, seq_len, n_channels)
        batch_size = x.size(0)
        
        # Transpose to (batch, n_channels, seq_len)
        x = x.transpose(1, 2)
        
        # Create patches: (batch, n_channels, n_patches, patch_len)
        patches = x.unfold(dimension=2, size=self.patch_len, step=self.stride)
        
        # Embed patches: (batch, n_channels, n_patches, d_model)
        x = self.patch_embed(patches)
        
        # Add positional and channel embeddings
        x = x + self.pos_embed + self.channel_embed
        
        # Reshape for transformer: (batch * n_channels, n_patches, d_model)
        x = x.reshape(batch_size * self.n_channels, self.n_patches, -1)
        
        # Transformer
        x = self.transformer(x)
        x = self.norm(x)
        
        # Take last patch representation (causal)
        x = x[:, -1, :]  # (batch * n_channels, d_model)
        
        # Reshape back: (batch, n_channels * d_model)
        x = x.reshape(batch_size, -1)
        
        # Classification
        x = self.dropout(x)
        return self.fc(x)

# Create PatchTST
patchtst = PatchTST(n_components, n_classes, window_size, 
                    patch_len=16, stride=8, d_model=32, nhead=4, num_layers=2)
patchtst_params = sum(p.numel() for p in patchtst.parameters())
patchtst_size_mb = patchtst_params * 4 / (1024 * 1024)
print(f"PatchTST parameters: {patchtst_params:,}")
print(f"PatchTST size: {patchtst_size_mb:.3f} MB")

In [ ]:
# Pretrained EEGNet - with compatibility workaround
HAS_PRETRAINED_EEGNET = False

try:
    # Suppress pandas warnings during braindecode import
    import warnings
    warnings.filterwarnings('ignore')
    
    from huggingface_hub import hf_hub_download
    
    # Try importing braindecode's EEGNetv4
    try:
        from braindecode.models import EEGNetv4
        BRAINDECODE_AVAILABLE = True
    except (TypeError, ImportError) as e:
        print(f"braindecode import issue (pandas compatibility): {e}")
        print("Using custom EEGNet implementation instead")
        BRAINDECODE_AVAILABLE = False
    
    warnings.filterwarnings('default')

except Exception as e:
    print(f"Setup error: {e}")
    BRAINDECODE_AVAILABLE = False

class PretrainedEEGNet(nn.Module):
    """
    EEGNet with pretrained weights, adapted for our channel count.
    Falls back to our custom EEGNet if braindecode has issues.
    """
    
    def __init__(self, n_channels_in, n_classes, n_times, freeze_backbone=False):
        super().__init__()
        
        self.n_channels_in = n_channels_in
        self.pretrained = False
        
        if BRAINDECODE_AVAILABLE:
            try:
                # Load pretrained weights
                path = hf_hub_download(
                    repo_id='PierreGtch/EEGNetv4',
                    filename='EEGNetv4_Lee2019_MI/model-params.pkl'
                )
                
                # Create model with pretrained config
                self.backbone = EEGNetv4(n_chans=22, n_outputs=2, n_times=1000)
                self.backbone.load_state_dict(torch.load(path, map_location='cpu', weights_only=False))
                print("Loaded pretrained EEGNet weights from HuggingFace")
                self.pretrained = True
                
                # Get feature size
                with torch.no_grad():
                    dummy = torch.zeros(1, 1, 22, 1000)
                    modules = list(self.backbone.children())
                    x = dummy
                    for module in modules[:-1]:
                        x = module(x)
                    feature_size = x.flatten(1).shape[1]
                
                self.classifier = nn.Linear(feature_size, n_classes)
                
                if freeze_backbone:
                    for param in self.backbone.parameters():
                        param.requires_grad = False
                        
            except Exception as e:
                print(f"Could not load pretrained weights: {e}")
                self.pretrained = False
        
        # Fallback: use our custom EEGNet (from earlier in notebook)
        if not self.pretrained:
            print("Using custom EEGNet (from scratch)")
            self.backbone = EEGNetModule(n_channels_in, n_classes, n_times)
            self.classifier = None
    
    def forward(self, x):
        if not self.pretrained:
            return self.backbone(x)
        
        # x: (batch, time, channels)
        batch_size = x.size(0)
        x = x.transpose(1, 2).unsqueeze(1)  # (batch, 1, channels, time)
        
        # Take first 22 channels for pretrained model
        x = x[:, :, :22, :]
        
        # Pad/crop time dimension
        if x.size(3) < 1000:
            x = F.pad(x, (0, 1000 - x.size(3)))
        elif x.size(3) > 1000:
            x = x[:, :, :, :1000]
        
        # Get features
        modules = list(self.backbone.children())
        for module in modules[:-1]:
            x = module(x)
        x = x.flatten(1)
        
        return self.classifier(x)

# Create model
try:
    pretrained_eegnet = PretrainedEEGNet(n_components, n_classes, window_size, freeze_backbone=False)
    pretrained_eegnet_params = sum(p.numel() for p in pretrained_eegnet.parameters())
    pretrained_eegnet_size_mb = pretrained_eegnet_params * 4 / (1024 * 1024)
    print(f"Pretrained EEGNet parameters: {pretrained_eegnet_params:,}")
    print(f"Pretrained EEGNet size: {pretrained_eegnet_size_mb:.3f} MB")
    HAS_PRETRAINED_EEGNET = True
except Exception as e:
    print(f"Could not create pretrained EEGNet: {e}")
    HAS_PRETRAINED_EEGNET = False

In [ ]:
from braindecode.models import EEGNetv4
from huggingface_hub import hf_hub_download

class PretrainedEEGNet(nn.Module):
    """
    EEGNet with pretrained weights, adapted for our channel count.
    
    Strategy: Load pretrained EEGNet, replace input and output layers
    to match our data dimensions, freeze middle layers initially.
    """
    
    def __init__(self, n_channels_in, n_classes, n_times, freeze_backbone=False):
        super().__init__()
        
        self.n_channels_in = n_channels_in
        
        # Load pretrained EEGNet (trained on 22 channels, 2 classes, 1000 samples)
        # We'll adapt the architecture for our data
        pretrained_channels = 22
        pretrained_classes = 2
        pretrained_times = 1000
        
        try:
            # Try to load pretrained weights
            path = hf_hub_download(
                repo_id='PierreGtch/EEGNetv4',
                filename='EEGNetv4_Lee2019_MI/model-params.pkl'
            )
            self.backbone = EEGNetv4(
                n_chans=pretrained_channels,
                n_outputs=pretrained_classes,
                n_times=pretrained_times
            )
            self.backbone.load_state_dict(torch.load(path, map_location='cpu', weights_only=False))
            print("Loaded pretrained EEGNet weights from HuggingFace")
            self.pretrained = True
        except Exception as e:
            print(f"Could not load pretrained weights: {e}")
            print("Using randomly initialized EEGNet")
            self.backbone = EEGNetv4(
                n_chans=n_channels_in,
                n_outputs=n_classes,
                n_times=n_times
            )
            self.pretrained = False
            return
        
        # Input adapter: project from our channels to pretrained channels
        self.input_adapter = nn.Conv2d(1, 1, (n_channels_in - pretrained_channels + 1, 1))
        
        # Output adapter: new classification head
        # Find the final layer size
        with torch.no_grad():
            dummy = torch.zeros(1, 1, pretrained_channels, pretrained_times)
            # Get features before final layer
            features = self._get_backbone_features(dummy)
            feature_size = features.shape[1]
        
        self.classifier = nn.Linear(feature_size, n_classes)
        
        # Optionally freeze backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            print("Backbone frozen - only training adapter and classifier")
    
    def _get_backbone_features(self, x):
        """Extract features from backbone before final classification."""
        # EEGNetv4 structure: we need to get features before the final linear layer
        # This depends on braindecode's implementation
        modules = list(self.backbone.children())
        for module in modules[:-1]:  # All except final linear
            x = module(x)
        return x.flatten(start_dim=1)
    
    def forward(self, x):
        # x: (batch, time, channels)
        batch_size = x.size(0)
        
        if not self.pretrained:
            # If not pretrained, use backbone directly
            x = x.transpose(1, 2).unsqueeze(1)  # (batch, 1, channels, time)
            return self.backbone(x)
        
        # Adapt input: (batch, time, n_channels_in) -> (batch, 1, 22, time)
        x = x.transpose(1, 2).unsqueeze(1)  # (batch, 1, n_channels_in, time)
        
        # Simple channel reduction via linear projection
        # Reshape to apply linear across channels
        x = x.squeeze(1)  # (batch, n_channels_in, time)
        x = x[:, :22, :]  # Take first 22 channels (simple approach)
        x = x.unsqueeze(1)  # (batch, 1, 22, time)
        
        # Pad time dimension if needed
        if x.size(3) < 1000:
            pad_size = 1000 - x.size(3)
            x = F.pad(x, (0, pad_size))
        elif x.size(3) > 1000:
            x = x[:, :, :, :1000]
        
        # Get features from backbone
        features = self._get_backbone_features(x)
        
        # Classify
        return self.classifier(features)

# Try to create pretrained model
try:
    pretrained_eegnet = PretrainedEEGNet(n_components, n_classes, window_size, freeze_backbone=False)
    pretrained_eegnet_params = sum(p.numel() for p in pretrained_eegnet.parameters())
    pretrained_eegnet_size_mb = pretrained_eegnet_params * 4 / (1024 * 1024)
    print(f"Pretrained EEGNet parameters: {pretrained_eegnet_params:,}")
    print(f"Pretrained EEGNet size: {pretrained_eegnet_size_mb:.3f} MB")
    HAS_PRETRAINED_EEGNET = True
except Exception as e:
    print(f"Could not create pretrained EEGNet: {e}")
    HAS_PRETRAINED_EEGNET = False

In [ ]:
# LaBraM is a large model and has braindecode compatibility issues
# Skip it for now and focus on models that work
HAS_LABRAM = False
print("Skipping LaBraM due to braindecode/pandas compatibility issues")
print("LaBraM would be ~100MB+ anyway, exceeding hackathon limits")

In [ ]:
class PretrainedLaBraM(nn.Module):
    """
    LaBraM foundation model with pretrained weights.
    
    LaBraM is a transformer-based model pretrained on large-scale EEG data.
    We add a classification head for our task.
    """
    
    def __init__(self, n_channels, n_classes, n_times, freeze_backbone=True):
        super().__init__()
        
        try:
            from braindecode.models import Labram
            
            # Load pretrained LaBraM
            self.backbone = Labram.from_pretrained("braindecode/labram-pretrained")
            print("Loaded pretrained LaBraM from HuggingFace")
            self.pretrained = True
            
            # LaBraM outputs features - add classification head
            # Get feature dimension
            with torch.no_grad():
                # LaBraM expects (batch, channels, time)
                dummy = torch.zeros(1, n_channels, n_times)
                try:
                    features = self.backbone(dummy)
                    if isinstance(features, tuple):
                        features = features[0]
                    feature_dim = features.shape[-1]
                except Exception as e:
                    print(f"Feature extraction failed: {e}")
                    feature_dim = 768  # Default LaBraM hidden size
            
            self.classifier = nn.Sequential(
                nn.LayerNorm(feature_dim),
                nn.Linear(feature_dim, n_classes)
            )
            
            if freeze_backbone:
                for param in self.backbone.parameters():
                    param.requires_grad = False
                print("LaBraM backbone frozen")
                
        except Exception as e:
            print(f"Could not load LaBraM: {e}")
            print("LaBraM requires specific braindecode version. Skipping.")
            self.pretrained = False
            # Fallback to simple model
            self.backbone = nn.Sequential(
                nn.Linear(n_channels * n_times, 256),
                nn.ReLU(),
                nn.Linear(256, n_classes)
            )
            self.classifier = None
    
    def forward(self, x):
        # x: (batch, time, channels) -> (batch, channels, time)
        x = x.transpose(1, 2)
        
        if not self.pretrained:
            x = x.flatten(1)
            return self.backbone(x)
        
        features = self.backbone(x)
        if isinstance(features, tuple):
            features = features[0]
        
        # Pool over time if needed
        if features.dim() == 3:
            features = features.mean(dim=1)
        
        return self.classifier(features)

# Try to create LaBraM model
try:
    labram_model = PretrainedLaBraM(n_components, n_classes, window_size, freeze_backbone=True)
    labram_params = sum(p.numel() for p in labram_model.parameters() if p.requires_grad)
    labram_total_params = sum(p.numel() for p in labram_model.parameters())
    labram_size_mb = labram_total_params * 4 / (1024 * 1024)
    print(f"LaBraM trainable parameters: {labram_params:,}")
    print(f"LaBraM total size: {labram_size_mb:.3f} MB")
    HAS_LABRAM = labram_model.pretrained
except Exception as e:
    print(f"Could not create LaBraM: {e}")
    HAS_LABRAM = False

### Model 8: Pretrained PatchTST (from HuggingFace)

PatchTST pretrained on time series data, adapted for classification.

Source: [ibm-granite/granite-timeseries-patchtst](https://huggingface.co/ibm-granite/granite-timeseries-patchtst)

In [ ]:
class PretrainedPatchTST(nn.Module):
    """
    PatchTST with pretrained weights from HuggingFace.
    
    Uses the transformers library to load pretrained PatchTST
    and adapts it for classification.
    """
    
    def __init__(self, n_channels, n_classes, n_times):
        super().__init__()
        
        try:
            from transformers import PatchTSTConfig, PatchTSTForClassification
            
            # Configure for our data
            config = PatchTSTConfig(
                num_input_channels=n_channels,
                context_length=n_times,
                patch_length=16,
                stride=8,
                num_targets=n_classes,
                d_model=64,  # Smaller for efficiency
                num_attention_heads=4,
                num_hidden_layers=2,
                ffn_dim=128,
                dropout=0.1,
            )
            
            self.model = PatchTSTForClassification(config)
            print(f"Created PatchTST for classification with config")
            self.pretrained = True
            
            # Note: For true pretrained weights, you would do:
            # self.model = PatchTSTForClassification.from_pretrained(
            #     "ibm/patchtst-etth1-pretrain",
            #     num_input_channels=n_channels,
            #     num_targets=n_classes,
            #     ignore_mismatched_sizes=True
            # )
            # But the pretrained model is for forecasting, not classification
            
        except ImportError as e:
            print(f"transformers library issue: {e}")
            print("Install with: pip install transformers")
            self.pretrained = False
            self.model = nn.Sequential(
                nn.Linear(n_channels * n_times, 256),
                nn.ReLU(),
                nn.Linear(256, n_classes)
            )
        except Exception as e:
            print(f"Could not create PatchTST: {e}")
            self.pretrained = False
            self.model = nn.Sequential(
                nn.Linear(n_channels * n_times, 256),
                nn.ReLU(),
                nn.Linear(256, n_classes)
            )
    
    def forward(self, x):
        # x: (batch, time, channels)
        if not self.pretrained:
            return self.model(x.flatten(1))
        
        # PatchTST expects (batch, channels, time)
        x = x.transpose(1, 2)
        output = self.model(past_values=x)
        return output.prediction_logits

# Try to create pretrained PatchTST
try:
    pretrained_patchtst = PretrainedPatchTST(n_components, n_classes, window_size)
    pretrained_patchtst_params = sum(p.numel() for p in pretrained_patchtst.parameters())
    pretrained_patchtst_size_mb = pretrained_patchtst_params * 4 / (1024 * 1024)
    print(f"PatchTST (HF) parameters: {pretrained_patchtst_params:,}")
    print(f"PatchTST (HF) size: {pretrained_patchtst_size_mb:.3f} MB")
    HAS_PRETRAINED_PATCHTST = pretrained_patchtst.pretrained
except Exception as e:
    print(f"Could not create pretrained PatchTST: {e}")
    HAS_PRETRAINED_PATCHTST = False

In [ ]:
models = {
    'MLP': (mlp, mlp_params, mlp_size_mb),
    'CNN': (cnn, cnn_params, cnn_size_mb),
    'Transformer': (transformer, trans_params, trans_size_mb),
    'EEGNet': (eegnet, eegnet_params, eegnet_size_mb),
    'PatchTST': (patchtst, patchtst_params, patchtst_size_mb),
}

print("Model Size Comparison:")
print("-" * 60)
print(f"{'Model':<15} {'Parameters':>15} {'Size (MB)':>15} {'Type':>15}")
print("-" * 60)
for name, (model, params, size) in models.items():
    model_type = "From scratch" if name in ['MLP', 'CNN', 'Transformer'] else "Established"
    print(f"{name:<15} {params:>15,} {size:>15.3f} {model_type:>15}")
print("-" * 60)
print(f"\nTarget: <5MB for optimal score, <25MB max")

In [ ]:
models = {
    'MLP': (mlp, mlp_params, mlp_size_mb),
    'CNN': (cnn, cnn_params, cnn_size_mb),
    'Transformer': (transformer, trans_params, trans_size_mb),
}

print("Model Size Comparison:")
print("-" * 50)
print(f"{'Model':<15} {'Parameters':>15} {'Size (MB)':>15}")
print("-" * 50)
for name, (model, params, size) in models.items():
    print(f"{name:<15} {params:>15,} {size:>15.3f}")
print("-" * 50)
print(f"\nTarget: <5MB for optimal score, <25MB max")

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=20, batch_size=128, lr=1e-3):
    """Train a model and return training history."""
    
    # Convert labels to indices
    classes = np.unique(y_train)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_train_idx = np.array([class_to_idx[y] for y in y_train])
    y_val_idx = np.array([class_to_idx[y] for y in y_val])
    
    # Create tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_idx, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_idx, dtype=torch.long)
    
    # Data loaders
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    # Validation loader to avoid OOM
    val_ds = torch.utils.data.TensorDataset(X_val_t, y_val_t)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        # Validation - batched to avoid OOM
        model.eval()
        all_preds = []
        with torch.no_grad():
            for X_batch, _ in val_loader:
                X_batch = X_batch.to(device)
                output = model(X_batch)
                preds = output.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
        
        val_acc = balanced_accuracy_score(y_val_idx, np.array(all_preds))
        
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {history['train_loss'][-1]:.4f}, Val Acc: {val_acc:.4f}")
        
        # Clear cache to prevent OOM buildup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return model, history, classes

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=20, batch_size=128, lr=1e-3):
    """Train a model and return training history."""
    
    # Convert labels to indices
    classes = np.unique(y_train)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y_train_idx = np.array([class_to_idx[y] for y in y_train])
    y_val_idx = np.array([class_to_idx[y] for y in y_val])
    
    # Create tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_idx, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_idx, dtype=torch.long)
    
    # Data loaders
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        # Validation
        model.eval()
        with torch.no_grad():
            X_val_t_dev = X_val_t.to(device)
            val_output = model(X_val_t_dev)
            val_preds = val_output.argmax(dim=1).cpu().numpy()
            val_acc = balanced_accuracy_score(y_val_idx, val_preds)
        
        history['train_loss'].append(total_loss / len(train_loader))
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {history['train_loss'][-1]:.4f}, Val Acc: {val_acc:.4f}")
    
    return model, history, classes

## Train and Compare Models

In [ ]:
# Train MLP on PCA features (no windowing needed)
print("Training MLP...")
mlp_model = SimpleMLP(n_components, 256, n_classes)
mlp_trained, mlp_history, classes = train_model(
    mlp_model, X_train_pca, y_train, X_val_pca, y_val, 
    epochs=20, batch_size=256
)

In [ ]:
# Train CNN on windowed data
print("\nTraining CNN...")
cnn_model = SmallCNN(n_components, n_classes, window_size)
cnn_trained, cnn_history, _ = train_model(
    cnn_model, X_train_win, y_train_win, X_val_win, y_val_win,
    epochs=20, batch_size=128
)

## Train Pretrained Models (Fine-tuning)

In [ ]:
# Train Pretrained EEGNet (fine-tuning)
if HAS_PRETRAINED_EEGNET:
    print("\nFine-tuning Pretrained EEGNet...")
    pretrained_eegnet_model = PretrainedEEGNet(n_components, n_classes, window_size, freeze_backbone=False)
    pretrained_eegnet_trained, pretrained_eegnet_history, _ = train_model(
        pretrained_eegnet_model, X_train_win, y_train_win, X_val_win, y_val_win,
        epochs=20, batch_size=128
    )
else:
    print("Skipping pretrained EEGNet (not available)")
    pretrained_eegnet_history = None

In [ ]:
# Train LaBraM (fine-tuning classification head)
if HAS_LABRAM:
    print("\nFine-tuning LaBraM...")
    labram_train_model = PretrainedLaBraM(n_components, n_classes, window_size, freeze_backbone=True)
    labram_trained, labram_history, _ = train_model(
        labram_train_model, X_train_win, y_train_win, X_val_win, y_val_win,
        epochs=20, batch_size=64  # Smaller batch for large model
    )
else:
    print("Skipping LaBraM (not available)")
    labram_history = None

In [ ]:
# Train Pretrained PatchTST
if HAS_PRETRAINED_PATCHTST:
    print("\nTraining PatchTST (HuggingFace)...")
    pretrained_patchtst_model = PretrainedPatchTST(n_components, n_classes, window_size)
    pretrained_patchtst_trained, pretrained_patchtst_history, _ = train_model(
        pretrained_patchtst_model, X_train_win, y_train_win, X_val_win, y_val_win,
        epochs=20, batch_size=64
    )
else:
    print("Skipping pretrained PatchTST (not available)")
    pretrained_patchtst_history = None

In [ ]:
# Train EEGNet on windowed data
print("\nTraining EEGNet...")
eegnet_model = EEGNetModule(n_components, n_classes, window_size, F1=8, D=2, F2=16, kernel_length=32)
eegnet_trained, eegnet_history, _ = train_model(
    eegnet_model, X_train_win, y_train_win, X_val_win, y_val_win,
    epochs=20, batch_size=128
)

In [ ]:
# Train PatchTST on windowed data
print("\nTraining PatchTST...")
patchtst_model = PatchTST(n_components, n_classes, window_size, 
                          patch_len=16, stride=8, d_model=32, nhead=4, num_layers=2)
patchtst_trained, patchtst_history, _ = train_model(
    patchtst_model, X_train_win, y_train_win, X_val_win, y_val_win,
    epochs=20, batch_size=128
)

In [ ]:
# Train Transformer on windowed data
print("\nTraining Transformer...")
trans_model = TinyTransformer(n_components, n_classes, d_model=64, nhead=4, num_layers=2)
trans_trained, trans_history, _ = train_model(
    trans_model, X_train_win, y_train_win, X_val_win, y_val_win,
    epochs=20, batch_size=128
)

In [ ]:
# Plot training curves for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(mlp_history['train_loss'], label='MLP', linestyle='-')
ax.plot(cnn_history['train_loss'], label='CNN', linestyle='-')
ax.plot(trans_history['train_loss'], label='Transformer', linestyle='-')
ax.plot(eegnet_history['train_loss'], label='EEGNet', linestyle='--')
ax.plot(patchtst_history['train_loss'], label='PatchTST', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss')
ax.legend()

# Validation accuracy
ax = axes[1]
ax.plot(mlp_history['val_acc'], label='MLP', linestyle='-')
ax.plot(cnn_history['val_acc'], label='CNN', linestyle='-')
ax.plot(trans_history['val_acc'], label='Transformer', linestyle='-')
ax.plot(eegnet_history['val_acc'], label='EEGNet', linestyle='--')
ax.plot(patchtst_history['val_acc'], label='PatchTST', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final comparison table
print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"{'Model':<15} {'Val Accuracy':>15} {'Size (MB)':>12} {'Window':>10} {'Type':>18}")
print("-"*80)

results = [
    ('MLP', mlp_history, mlp_size_mb, 'N/A', 'From scratch'),
    ('CNN', cnn_history, cnn_size_mb, f'{window_size}ms', 'From scratch'),
    ('Transformer', trans_history, trans_size_mb, f'{window_size}ms', 'From scratch'),
    ('EEGNet', eegnet_history, eegnet_size_mb, f'{window_size}ms', 'Established (BCI)'),
    ('PatchTST', patchtst_history, patchtst_size_mb, f'{window_size}ms', 'Established (TS)'),
]

# Sort by accuracy
results_sorted = sorted(results, key=lambda x: x[1]['val_acc'][-1], reverse=True)

for name, hist, size, window, mtype in results_sorted:
    acc = hist['val_acc'][-1] * 100
    print(f"{name:<15} {acc:>14.1f}% {size:>12.3f} {window:>10} {mtype:>18}")

print("-"*80)
print(f"\nNote: Window size adds inherent lag to predictions.")
print("For hackathon scoring, lag <100ms is optimal, model <5MB is optimal.")

In [ ]:
# Final comparison
print("\n" + "="*70)
print("FINAL RESULTS")
print("="*70)
print(f"{'Model':<15} {'Val Accuracy':>15} {'Size (MB)':>15} {'Window (ms)':>15}")
print("-"*70)
print(f"{'MLP':<15} {mlp_history['val_acc'][-1]*100:>14.1f}% {mlp_size_mb:>15.3f} {'N/A':>15}")
print(f"{'CNN':<15} {cnn_history['val_acc'][-1]*100:>14.1f}% {cnn_size_mb:>15.3f} {window_size:>15}")
print(f"{'Transformer':<15} {trans_history['val_acc'][-1]*100:>14.1f}% {trans_size_mb:>15.3f} {window_size:>15}")
print("-"*70)
print(f"\nNote: Window size of {window_size}ms adds inherent lag to predictions.")
print("For the hackathon scoring, lag <100ms is optimal.")

In [ ]:
def measure_inference_time(model, sample_input, n_runs=1000):
    """Measure average inference time per sample."""
    model.eval()
    model = model.to('cpu')  # Test on CPU for fair comparison
    
    sample = torch.tensor(sample_input, dtype=torch.float32).unsqueeze(0)
    
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(sample)
    
    # Measure
    start = time.perf_counter()
    for _ in range(n_runs):
        with torch.no_grad():
            _ = model(sample)
    end = time.perf_counter()
    
    return (end - start) / n_runs * 1000  # ms

print("Inference Time (per sample on CPU):")
print("-" * 40)

mlp_time = measure_inference_time(mlp_trained, X_val_pca[0])
cnn_time = measure_inference_time(cnn_trained, X_val_win[0])
trans_time = measure_inference_time(trans_trained, X_val_win[0])
eegnet_time = measure_inference_time(eegnet_trained, X_val_win[0])
patchtst_time = measure_inference_time(patchtst_trained, X_val_win[0])

timings = [
    ('MLP', mlp_time),
    ('CNN', cnn_time),
    ('Transformer', trans_time),
    ('EEGNet', eegnet_time),
    ('PatchTST', patchtst_time),
]

for name, t in sorted(timings, key=lambda x: x[1]):
    print(f"  {name:<15}: {t:.3f} ms")

print("-" * 40)
print("Target: <1ms per sample for real-time at 1000Hz")

## Recommendations

### Model Comparison Summary

| Model | Origin | Best For |
|-------|--------|----------|
| **MLP** | From scratch | Fast baseline, no temporal context needed |
| **CNN** | From scratch | Good balance of size and accuracy |
| **Transformer** | From scratch | Strong sequence modeling |
| **EEGNet** | Established (BCI) | Designed for neural signals, very compact |
| **PatchTST** | Established (TS) | State-of-art time series, patch-based |

### Hackathon Strategy

1. **For best overall score**: EEGNet or small CNN
   - Tiny model size = excellent size score
   - Designed for neural data (EEGNet)
   - Window can be tuned to minimize lag

2. **For max accuracy**: PatchTST or Transformer
   - Better at learning complex temporal patterns
   - Larger window = more context = better accuracy
   - Trade-off: more lag penalty

3. **For minimal lag**: MLP on current sample only
   - No window = no lag
   - Trade-off: loses temporal context

### Next Steps
- Try different window sizes (32, 64, 128, 256ms)
- Experiment with EEGNet hyperparameters (F1, D, kernel_length)
- Consider ensemble of fast + accurate models

## Recommendations

Based on the evaluation:

1. **For best hackathon score**: Use simple MLP or small CNN with minimal window size
   - Small model size = better size score
   - No/small window = lower lag

2. **If accuracy is priority**: Transformer with windowed input
   - Better at capturing temporal patterns
   - But window adds lag penalty

3. **Spectrogram approach**: Worth trying if frequency discrimination is key
   - Adds computational overhead
   - Window requirement adds lag

**Trade-off**: The scoring formula heavily penalizes both lag (>100ms) and model size (>5MB).
A simple, small model that's fast may score better than a complex accurate one.

In [ ]:
print("Notebook complete!")